In [1]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.optim.lr_scheduler import PolynomialLR
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

# Add project directories so local modules import correctly
repo_root = Path.cwd()
for p in (repo_root / "04_modeling", repo_root / "03_datapreparation"):
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.append(p_str)

# Models
from models.unet import UNet
# Utils
from utils.focal_loss import FocalLoss
# Dataset, DataLoader and Transforms
from cityscapes_dataset import CityscapesDataset
from torch.utils.data import DataLoader
import torchvision.transforms.functional as F
from PIL import Image
# For visualization, logging
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter


In [2]:
def compute_miou(pred, truth, num_classes, ignore_index=255):
    # TODO: Make computation run on GPU using torch 
    ious = []
    pred = pred.flatten()
    truth = truth.flatten()

    # Remove label values of ignore_index
    mask = truth != ignore_index
    pred = pred[mask]
    truth = truth[mask]

    for clas in range(num_classes):
        pred_clas = pred == clas
        truth_clas = truth == clas
        intersection = (pred_clas & truth_clas).sum().item()
        union = (pred_clas | truth_clas).sum().item()
        if union == 0:
            ious.append(float("nan"))
        else:
            ious.append(intersection/union)
    return np.nanmean(ious), ious

def compute_accuracy(pred, truth, ignore_index=255):
    # Remove label values of ignore_index
    mask = truth != ignore_index
    pred = pred[mask]
    truth = truth[mask]

    return (pred == truth).float().mean()

## Trainingsloop
In jedem Schritt:
- Eingabebilder und zugehörige Labels auf `device` laden
- Vorwärtsberechnung `forward pass` durchführen
- Verlust `loss` berechnen
- Gradienten mit `loss.backward()` berechnen
- Parameter mit `optimizer.step()` aktualisieren
- Metriken `loss`, `accuracy` und `MIoU` berechnen

In [3]:
def train(dataloader, model, loss_function, optimizer, device, num_classes, scheduler, min_lr):
    # Switch to train mode
    model.train()

    # For storing running metrics
    epoch_loss, epoch_acc, epoch_miou = 0, 0, 0

    # Create a progress bar
    pbar = tqdm(dataloader, desc="Training loop", leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        ### Forward pass
        # Get output/prediction
        output = model(images)
        # Calculate loss
        loss = loss_function(output, labels)

        ### Backpropagation
        # Gradients have to be zeroed for each batch
        optimizer.zero_grad()
        # Calculate the gradients
        loss.backward()
        # Update the parameters
        optimizer.step()

        ### LR Scheduler
        scheduler.step()

        ### Calculate metrics per batch
        prediction = output.argmax(dim=1)

        acc = compute_accuracy(prediction, labels)
        miou, _ = compute_miou(prediction, labels, num_classes)
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        epoch_miou += miou

    # Return metrics averaged across all batches
    num_batches = len(dataloader)
    return epoch_loss/num_batches, epoch_acc/num_batches, epoch_miou/num_batches

## Validationloop
In jedem Schritt:
- Eingabebilder und zugehörige Labels auf `device` laden
- Vorwärtsberechnung `forward pass` durchführen
- Verlust `loss` berechnen
- Metriken `loss`, `accuracy` und `MIoU` berechnen

In [4]:
def valid(dataloader, model, loss_function, device, num_classes):
    # Switch to eval mode
    model.eval()

    # For storing running metrics
    epoch_loss, epoch_acc, epoch_miou = 0, 0, 0

    # Create a progress bar
    pbar = tqdm(dataloader, desc="Validation loop", leave=False)
    # Disable gradient calculation
    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            ### Forward pass
            # Get output/prediction
            output = model(images)
            # Calculate loss
            loss = loss_function(output, labels)

            ### Calculate metrics per batch
            prediction = output.argmax(dim=1)

            acc = compute_accuracy(prediction, labels)
            miou, _ = compute_miou(prediction, labels, num_classes)
            epoch_loss += loss.item()
            epoch_acc += acc.item()
            epoch_miou += miou
    
    # Return metrics averaged across all batches
    num_batches = len(dataloader)
    return epoch_loss/num_batches, epoch_acc/num_batches, epoch_miou/num_batches
            
            

## Create the DataLoaders
- Transforms definieren (`Resize` zum Probieren, `RandomCrop` vlt. insgesamt sinnvoller)
- Pfade für Bilder und Annotationen angeben
- Datasets für `train` und `val` split initialisieren
- Dataloaders mit Dataset und Batchgröße `batch_size` initialisieren

In [5]:
### Transforms to resize images, labels
height, width = 512, 1024
def image_transform(image):
    image = Image.fromarray(image)  # numpy -> PIL
    image = F.resize(image, (height, width), interpolation=F.InterpolationMode.BILINEAR) # Bilinear for RGB-Images
    image = F.pil_to_tensor(image).float() / 255.0  # Scaling
    return image


def label_transform(label):
    label = Image.fromarray(label)  # numpy -> PIL
    label = F.resize(label, (height, width), interpolation=F.InterpolationMode.NEAREST) # Nearest Neighbor for Labels
    label = F.pil_to_tensor(label).long().squeeze()  # PIL -> LongTensor no Scaling, squeeze() to remove channel dimension
    return label


# Define data paths
data_dir = Path("./data/cityscapes_intersections_heading/fine_intersections_4")
image_dir = data_dir / "leftImg8bit_trainvaltest/leftImg8bit"
label_dir = data_dir / "gtFine_trainvaltest/gtFine"

# Initialize datasets
train_dataset = CityscapesDataset(image_dir, label_dir, split="train", image_transform=image_transform, label_transform=label_transform)
valid_dataset = CityscapesDataset(image_dir, label_dir, split="val", image_transform=image_transform, label_transform=label_transform)

from torch.utils.data import Subset
train_subset = Subset(train_dataset, indices=[0, 1])
valid_subset = Subset(valid_dataset, indices=[0, 1])

# Create the dataloaders
train_dataloader = DataLoader(train_subset, batch_size=2, shuffle=True)
valid_dataloader = DataLoader(valid_subset, batch_size=2, shuffle=False)

for images, labels in train_dataloader:
    print(f"Images of first batch: {images.shape}")
    print(f"Labels of first batch: {labels.shape}")
    break

for images, labels in valid_dataloader:
    print(f"Images of first batch: {images.shape}")
    print(f"Labels of first batch: {labels.shape}")
    print(labels.dtype)
    print(labels.unique())
    break

Images of first batch: torch.Size([2, 3, 512, 1024])
Labels of first batch: torch.Size([2, 512, 1024])
Images of first batch: torch.Size([2, 3, 512, 1024])
Labels of first batch: torch.Size([2, 512, 1024])
torch.int64
tensor([  0,   1,   3,   5,   6,   7,   8,   9,  10,  12,  14,  16, 255])


In [ ]:
### Hyperparameters
num_epochs = 25
learning_rate = 1e-4
min_lr = 1e-6

warmup_epochs = 10
num_epochs = 100
iters_per_epoch = len(train_dataloader)
warmup_iters = warmup_epochs * iters_per_epoch
total_iters = num_epochs * iters_per_epoch

focal_exponent = 1 # gamma for focal loss

iters_per_epoch = len(train_dataloader)
total_iters = num_epochs * iters_per_epoch

save_at_epochs = 500

# Path to save checkpoints
run = "warmup-cosine2"
directory = Path("./checkpoints") / run
directory.mkdir(parents=True, exist_ok=True)

# For logging to tensorboard
writer = SummaryWriter(log_dir=f"./logs/{run}")

# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Initialize model and move to device
num_classes = 18    # Cityscapes custom (0..17)
model = UNet(num_classes).to(device)
model_name = model.name

# Load checkpoint with torch.load (optionally)
loaded_checkpoint = None
loaded_epochs = 0

# Define optimizer and loss function
#loss_function = nn.CrossEntropyLoss(ignore_index=255)
loss_function = FocalLoss(gamma=focal_exponent, ignore_index=255)
optimizer = optim.Adam(params=model.parameters(), lr=learning_rate)

### Scheduler
#scheduler = PolynomialLR(
#    optimizer,
#    total_iters=total_iters,
#    power=power
#)

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=1e-2,      # startet bei 1%
    total_iters=warmup_iters
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_iters - warmup_iters,
    eta_min=min_lr
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_iters]
)

best_val_loss = float(100_000)

if loaded_checkpoint:
    model.load_state_dict(loaded_checkpoint["model_state_dict"])
    loaded_epochs = loaded_checkpoint["epoch"]
    optimizer.load_state_dict(loaded_checkpoint["optimizer_state_dict"])
    best_val_loss = loaded_checkpoint["val_loss"]
    
pbar = tqdm(range(num_epochs))
for epoch in pbar:
    pbar.set_description(f"Epoch {epoch+1}/{num_epochs}")

    # Call train and validation loops
    train_loss, train_acc, train_miou = train(train_dataloader, model, loss_function, optimizer, device, num_classes, scheduler, min_lr)
    val_loss, val_acc, val_miou = valid(valid_dataloader, model, loss_function, device, num_classes)
    
    checkpoint = {
        "epoch": loaded_epochs + epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "train_miou": train_miou,
        "val_miou": val_miou
    }

    # Save the model every {save_at_epochs} epochs
    if epoch % save_at_epochs == 0:
        torch.save(checkpoint, f"./checkpoints/{run}/{model_name}_epoch_{loaded_epochs + epoch}")

    # Save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(checkpoint, f"./checkpoints/{run}/best_{model_name}")
    
    # Update the progress bar with metrics from current epoch
    pbar.set_postfix({'Train Loss':f'{train_loss:.3f}', 'Val Loss':f'{val_loss:.3f}', 'Train Acc':f'{train_acc:.3f}', 'Val Acc':f'{val_acc:.3f}', 
                      "Train MIoU":f"{train_miou:.3f}", "Val MIoU":f"{val_miou:.3f}"})
    
    # TODO: Better logging to tensorboard
    # Usage: tensorboard --logdir={log_dir}
    current_lr = optimizer.param_groups[0]["lr"]
    writer.add_scalar("lr", current_lr, global_step=epoch)
    writer.add_scalars("train/val loss", {"train loss": train_loss, "val loss": val_loss}, global_step=epoch)
    writer.add_scalars("train/val acc", {"train acc": train_acc, "val_acc": val_acc}, global_step=epoch)
    writer.add_scalars("train/val MIoU", {"train MIoU": train_miou, "val_miou": val_miou}, global_step=epoch)
    writer.flush()

writer.close()

Device: cuda


Epoch 100/100: 100%|██████████| 100/100 [04:30<00:00,  2.71s/it, Train Loss=1.058, Val Loss=1.419, Train Acc=0.410, Val Acc=0.365, Train MIoU=0.049, Val MIoU=0.045]
